# Build4All — Policy-Dependent Fresh Test (PD-Test)

**Run this only after Claude says the controllers are frozen.** This set was never used for
training, tuning, or controller design. Every system below is frozen; nothing may be changed after
the results are seen.

Before starting, upload `build4all_policy_test_v1.csv` (the version you reviewed) to
`MyDrive/Build4All_Research/Revision_Experiments/`.

## Part A — Setup (same as the revision notebook; run all after every reconnect)

In [1]:
# A1 — Mount Drive and locate inputs
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import torch, random, numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

ROOT = Path("/content/drive/MyDrive/Build4All_Research")
REV = ROOT / "Revision_Experiments"
REV.mkdir(parents=True, exist_ok=True)

dataset_candidates = sorted(ROOT.rglob("build4all_multilingual_text2sql_600.csv"))
assert dataset_candidates, "Upload build4all_multilingual_text2sql_600.csv to Build4All_Research/Revision_Experiments/"
DATASET_CSV = dataset_candidates[0]

adapter_candidates = sorted(
    p.parent for p in ROOT.rglob("adapter_model.safetensors")
    if p.parent.name == "qwen3_8b_build4all_strict_qlora_adapter"
)
assert adapter_candidates, "Strict QLoRA adapter folder not found on Drive."
ADAPTER_DIR = adapter_candidates[0]
assert (ADAPTER_DIR / "adapter_config.json").exists(), f"adapter_config.json missing in {ADAPTER_DIR}"

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Select a T4 GPU runtime."
print("Dataset:", DATASET_CSV)
print("Strict adapter:", ADAPTER_DIR)
print("Results folder:", REV)

Mounted at /content/drive
GPU: Tesla T4
Dataset: /content/drive/MyDrive/Build4All_Research/Database/Datasets/build4all_multilingual_text2sql_600.csv
Strict adapter: /content/drive/MyDrive/Build4All_Research/Final_Test/Extracted/build4all_final_experiment/fine_tuning/qwen3_8b_build4all_strict_qlora_adapter
Results folder: /content/drive/MyDrive/Build4All_Research/Revision_Experiments


In [2]:
# ============================================================
# CELL 2 — INSTALL AND VALIDATE REQUIRED DEPENDENCIES
# ============================================================

%pip install --quiet \
    "bitsandbytes>=0.46.1" \
    "faiss-cpu>=1.8.0" \
    "pypdf>=5.0.0" \
    "psycopg2-binary>=2.9.9" \
    "sqlglot>=25.0.0" \
    "sentence-transformers>=3.0.0" \
    "peft>=0.17.0" \
    "accelerate>=1.0.0"

import bitsandbytes
import faiss
import pypdf
import psycopg2
import sqlglot
import sentence_transformers
import transformers

print("=" * 70)
print("DEPENDENCY VALIDATION")
print("=" * 70)

print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("FAISS ready:", hasattr(faiss, "IndexFlatIP"))
print("PyPDF ready:", hasattr(pypdf, "PdfReader"))
print("Psycopg2 ready:", hasattr(psycopg2, "connect"))
print("SQLGlot:", sqlglot.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)

assert hasattr(faiss, "IndexFlatIP")
assert hasattr(pypdf, "PdfReader")
assert hasattr(psycopg2, "connect")

print("\nCELL 2 COMPLETED SUCCESSFULLY")
print("Qwen loaded: False")
print("Database connected: False")
print("Generations performed: 0")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 116.0 MB/s eta 0:00:00
DEPENDENCY VALIDATION
Transformers: 5.18.0
BitsAndBytes: 0.50.2
FAISS ready: True
PyPDF ready: True
Psycopg2 ready: True
SQLGlot: 25.20.2
Sentence Transformers: 5.7.0

CELL 2 COMPLETED SUCCESSFULLY
Qwen loaded: False
Database connected: False
Generations performed: 0


In [3]:
# ============================================================
# CELL 3 — FROZEN EXPERIMENT CONFIGURATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/Build4All_Research/Revision_Experiments/build4all_final_experiment")

PATHS = {
    "root": PROJECT_ROOT,
    "config": PROJECT_ROOT / "config",
    "data": PROJECT_ROOT / "data",
    "documents": PROJECT_ROOT / "documents",
    "indexes": PROJECT_ROOT / "indexes",
    "predictions": PROJECT_ROOT / "predictions",
    "controller": PROJECT_ROOT / "controller",
    "reports": PROJECT_ROOT / "reports",
    "checkpoints": PROJECT_ROOT / "checkpoints",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "project_name": "Build4All Qwen3 RAG Controller Final Experiment",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "seed": 42,

    "model": {
        "model_id": "Qwen/Qwen3-8B",
        "quantization": "4-bit NF4",
        "do_sample": False,
        "enable_thinking": False,
        "max_new_tokens": 512,
    },

    "database": {
        "provider": "Neon PostgreSQL",
        "schema": "build4all_research",
        "secret_name": "NEON_DATABASE_URL",
        "read_only": True,
        "statement_timeout_seconds": 30,
    },

    "languages": [
        "English",
        "Arabic",
        "French",
        "Lebanese Arabizi",
    ],

    "rag": {
        "embedding_model": "intfloat/multilingual-e5-base",
        "retrieval_type": "dense",
        "index_type": "FAISS IndexFlatIP",
        "top_k": 4,
        "max_context_characters": 4000,
        "normalized_embeddings": True,
    },

    "experiment": {
        "systems": [
            "Qwen Alone",
            "Qwen + RAG",
            "Adaptive Controller",
        ],
        "target_question_count": 600,
        "split_method": "semantic-family group split",
        "train_ratio": 0.60,
        "validation_ratio": 0.20,
        "final_test_ratio": 0.20,
        "primary_metric": "execution accuracy",
        "minimum_target_accuracy": 0.75,
        "final_test_locked": True,
    },
}

config_path = PATHS["config"] / "experiment_config.json"

with config_path.open("w", encoding="utf-8") as file:
    json.dump(CONFIG, file, indent=2, ensure_ascii=False)

assert CONFIG["model"]["do_sample"] is False
assert CONFIG["database"]["read_only"] is True
assert CONFIG["experiment"]["final_test_locked"] is True
assert sum([
    CONFIG["experiment"]["train_ratio"],
    CONFIG["experiment"]["validation_ratio"],
    CONFIG["experiment"]["final_test_ratio"],
]) == 1.0

print("=" * 70)
print("FROZEN EXPERIMENT CONFIGURATION")
print("=" * 70)

print("Model:", CONFIG["model"]["model_id"])
print("Database schema:", CONFIG["database"]["schema"])
print("Languages:", CONFIG["languages"])
print("Systems:", CONFIG["experiment"]["systems"])
print("Target questions:", CONFIG["experiment"]["target_question_count"])
print("Minimum target accuracy:", CONFIG["experiment"]["minimum_target_accuracy"])
print("Final test locked:", CONFIG["experiment"]["final_test_locked"])
print("Configuration saved:", config_path)

print("\nCELL 3 COMPLETED SUCCESSFULLY")
print("Qwen loaded: False")
print("Database connected: False")
print("Generations performed: 0")

FROZEN EXPERIMENT CONFIGURATION
Model: Qwen/Qwen3-8B
Database schema: build4all_research
Languages: ['English', 'Arabic', 'French', 'Lebanese Arabizi']
Systems: ['Qwen Alone', 'Qwen + RAG', 'Adaptive Controller']
Target questions: 600
Minimum target accuracy: 0.75
Final test locked: True
Configuration saved: /content/drive/MyDrive/Build4All_Research/Revision_Experiments/build4all_final_experiment/config/experiment_config.json

CELL 3 COMPLETED SUCCESSFULLY
Qwen loaded: False
Database connected: False
Generations performed: 0


In [5]:
# ============================================================
# CELL 4 — SECURE NEON CONNECTION AND SCHEMA AUDIT
# ============================================================

from google.colab import userdata
import psycopg2
import pandas as pd

SECRET_NAME = CONFIG["database"]["secret_name"]
SCHEMA_NAME = CONFIG["database"]["schema"]

# ------------------------------------------------------------
# 1. Read Neon URL securely from Kaggle Secrets
# ------------------------------------------------------------

try:
    # Colab Secrets
    #connection URL men Kaggle Secrets.
    NEON_DATABASE_URL = userdata.get(SECRET_NAME)

except Exception as error:
    raise RuntimeError(
        "NEON_DATABASE_URL was not found or is not enabled."
    ) from error


# ------------------------------------------------------------
# 2. Create a reusable read-only connection
# ------------------------------------------------------------

def get_db_connection():

    connection = psycopg2.connect(
        NEON_DATABASE_URL,
        sslmode="require",
        connect_timeout=15,
        application_name="build4all_final_experiment",
        options="-c default_transaction_read_only=on",
    )

    # The Neon pooler accepts statement_timeout after connection.
    with connection.cursor() as cursor:
        cursor.execute(
            "SET statement_timeout TO 30000;"
        )

    return connection


# ------------------------------------------------------------
# 3. Define the expected final tables
# ------------------------------------------------------------

expected_tables = {
    "categories",
    "coupons",
    "customers",
    "shipping_methods",
    "tax_rules",
    "products",
    "orders",
    "order_items",
    "payments",
    "product_events",
}


# ------------------------------------------------------------
# 4. Test connection and inspect schema
# ------------------------------------------------------------

with get_db_connection() as connection:

    with connection.cursor() as cursor:

        cursor.execute("""
            SELECT
                current_database(),
                current_user,
                current_setting('transaction_read_only');
        """)

        database_name, database_user, read_only = (
            cursor.fetchone()
        )

        cursor.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = %s
              AND table_type = 'BASE TABLE'
            ORDER BY table_name;
        """, (SCHEMA_NAME,))

        discovered_tables = {
            row[0]
            for row in cursor.fetchall()
        }

        cursor.execute("""
            SELECT
                table_name,
                column_name,
                data_type
            FROM information_schema.columns
            WHERE table_schema = %s
            ORDER BY table_name, ordinal_position;
        """, (SCHEMA_NAME,))

        schema_rows = cursor.fetchall()


# ------------------------------------------------------------
# 5. Store the schema audit
# ------------------------------------------------------------

schema_audit = pd.DataFrame(
    schema_rows,
    columns=[
        "table_name",
        "column_name",
        "data_type",
    ],
)

missing_tables = (
    expected_tables - discovered_tables
)

unexpected_tables = (
    discovered_tables - expected_tables
)


# ------------------------------------------------------------
# 6. Display safe audit results
# ------------------------------------------------------------

print("=" * 70)
print("NEON CONNECTION AND SCHEMA AUDIT")
print("=" * 70)

print("Database:", database_name)
print("Database user:", database_user)
print("Read-only mode:", read_only)
print("Schema:", SCHEMA_NAME)
print("Table count:", len(discovered_tables))
print("Tables:", sorted(discovered_tables))
print("Missing tables:", sorted(missing_tables))
print("Unexpected tables:", sorted(unexpected_tables))

print("\nCOLUMN COUNT BY TABLE")

print(
    schema_audit
    .groupby("table_name")
    .size()
    .rename("column_count")
    .to_string()
)


# ------------------------------------------------------------
# 7. Stop if the database is incomplete or unsafe
# ------------------------------------------------------------

assert read_only == "on", (
    "Database connection is not read-only."
)

assert not missing_tables, (
    f"Missing tables: {sorted(missing_tables)}"
)

assert len(schema_audit) > 0, (
    "No schema columns were discovered."
)


print("\nCELL 4 COMPLETED SUCCESSFULLY")
print("Database connected: True")
print("Schema validated: True")
print("Qwen loaded: False")
print("Generations performed: 0")

NEON CONNECTION AND SCHEMA AUDIT
Database: neondb
Database user: neondb_owner
Read-only mode: on
Schema: build4all_research
Table count: 10
Tables: ['categories', 'coupons', 'customers', 'order_items', 'orders', 'payments', 'product_events', 'products', 'shipping_methods', 'tax_rules']
Missing tables: []
Unexpected tables: []

COLUMN COUNT BY TABLE
table_name
categories           3
coupons             10
customers            5
order_items          5
orders              11
payments             6
product_events       5
products            11
shipping_methods     7
tax_rules            7

CELL 4 COMPLETED SUCCESSFULLY
Database connected: True
Schema validated: True
Qwen loaded: False
Generations performed: 0


In [6]:
# ============================================================
# CELL 5 — EXACT DATABASE SCHEMA AND DATA VOCABULARY
# ============================================================

from psycopg2 import sql
import json

# ------------------------------------------------------------
# 1. Get columns
# ------------------------------------------------------------

with get_db_connection() as connection:
    with connection.cursor() as cursor:

        cursor.execute("""
            SELECT
                table_name,
                column_name,
                data_type,
                is_nullable
            FROM information_schema.columns
            WHERE table_schema = %s
            ORDER BY table_name, ordinal_position;
        """, (SCHEMA_NAME,))

        column_rows = cursor.fetchall()

        # Primary keys
        cursor.execute("""
            SELECT
                tc.table_name,
                kcu.column_name
            FROM information_schema.table_constraints tc
            JOIN information_schema.key_column_usage kcu
              ON tc.constraint_name = kcu.constraint_name
             AND tc.table_schema = kcu.table_schema
            WHERE tc.table_schema = %s
              AND tc.constraint_type = 'PRIMARY KEY'
            ORDER BY tc.table_name, kcu.ordinal_position;
        """, (SCHEMA_NAME,))

        primary_key_rows = cursor.fetchall()

        # Foreign keys
        cursor.execute("""
            SELECT
                tc.table_name,
                kcu.column_name,
                ccu.table_name AS referenced_table,
                ccu.column_name AS referenced_column
            FROM information_schema.table_constraints tc
            JOIN information_schema.key_column_usage kcu
              ON tc.constraint_name = kcu.constraint_name
             AND tc.table_schema = kcu.table_schema
            JOIN information_schema.constraint_column_usage ccu
              ON tc.constraint_name = ccu.constraint_name
             AND tc.table_schema = ccu.table_schema
            WHERE tc.table_schema = %s
              AND tc.constraint_type = 'FOREIGN KEY'
            ORDER BY tc.table_name, kcu.column_name;
        """, (SCHEMA_NAME,))

        foreign_key_rows = cursor.fetchall()

        # Row counts
        table_row_counts = {}

        for table_name in sorted(discovered_tables):
            cursor.execute(
                sql.SQL("SELECT COUNT(*) FROM {}.{}").format(
                    sql.Identifier(SCHEMA_NAME),
                    sql.Identifier(table_name),
                )
            )

            table_row_counts[table_name] = (
                cursor.fetchone()[0]
            )

        # Controlled categorical values
        vocabulary_queries = {
            "orders.status": """
                SELECT DISTINCT status
                FROM build4all_research.orders
                ORDER BY status
            """,

            "payments.payment_status": """
    SELECT DISTINCT payment_status
    FROM build4all_research.payments
    ORDER BY payment_status
""",

            "payments.payment_method": """
                SELECT DISTINCT payment_method
                FROM build4all_research.payments
                ORDER BY payment_method
            """,

            "products.status": """
                SELECT DISTINCT status
                FROM build4all_research.products
                ORDER BY status
            """,

            "product_events.event_type": """
                SELECT DISTINCT event_type
                FROM build4all_research.product_events
                ORDER BY event_type
            """,
        }

        controlled_vocabulary = {}

        for name, query in vocabulary_queries.items():
            cursor.execute(query)

            controlled_vocabulary[name] = [
                row[0]
                for row in cursor.fetchall()
            ]


# ------------------------------------------------------------
# 2. Build a compact schema description for Qwen
# ------------------------------------------------------------

primary_keys = {
    (table_name, column_name)
    for table_name, column_name in primary_key_rows
}

schema_lines = []

current_table = None

for table_name, column_name, data_type, nullable in column_rows:

    if table_name != current_table:
        if current_table is not None:
            schema_lines.append("")

        schema_lines.append(
            f"TABLE {SCHEMA_NAME}.{table_name}"
        )

        current_table = table_name

    key_label = (
        " PRIMARY KEY"
        if (table_name, column_name) in primary_keys
        else ""
    )

    null_label = (
        " NULL"
        if nullable == "YES"
        else " NOT NULL"
    )

    schema_lines.append(
        f"- {column_name}: {data_type}"
        f"{key_label}{null_label}"
    )

schema_lines.append("")
schema_lines.append("FOREIGN KEYS")

for (
    table_name,
    column_name,
    referenced_table,
    referenced_column,
) in foreign_key_rows:

    schema_lines.append(
        f"- {SCHEMA_NAME}.{table_name}.{column_name} "
        f"-> {SCHEMA_NAME}.{referenced_table}."
        f"{referenced_column}"
    )

DATABASE_SCHEMA_TEXT = "\n".join(schema_lines)


# ------------------------------------------------------------
# 3. Save the audited schema
# ------------------------------------------------------------

schema_text_path = (
    PATHS["data"] / "database_schema.txt"
)

schema_json_path = (
    PATHS["data"] / "database_audit.json"
)

schema_text_path.write_text(
    DATABASE_SCHEMA_TEXT,
    encoding="utf-8",
)

schema_json_path.write_text(
    json.dumps(
        {
            "schema": SCHEMA_NAME,
            "row_counts": table_row_counts,
            "controlled_vocabulary": controlled_vocabulary,
            "primary_keys": primary_key_rows,
            "foreign_keys": foreign_key_rows,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 4. Display audit
# ------------------------------------------------------------

print("=" * 70)
print("EXACT DATABASE SCHEMA")
print("=" * 70)
print(DATABASE_SCHEMA_TEXT)

print("\n" + "=" * 70)
print("TABLE ROW COUNTS")
print("=" * 70)

for table_name, row_count in table_row_counts.items():
    print(f"{table_name:<20}: {row_count}")

print("\n" + "=" * 70)
print("CONTROLLED VOCABULARY")
print("=" * 70)

for field_name, values in controlled_vocabulary.items():
    print(f"{field_name}: {values}")

assert all(
    row_count > 0
    for row_count in table_row_counts.values()
), "At least one table is empty."

print("\nCELL 5 COMPLETED SUCCESSFULLY")
print("Exact schema saved:", schema_text_path)
print("Database audit saved:", schema_json_path)
print("Qwen loaded: False")
print("Generations performed: 0")

EXACT DATABASE SCHEMA
TABLE build4all_research.categories
- category_id: integer PRIMARY KEY NOT NULL
- name: character varying NOT NULL
- parent_category_id: integer NULL

TABLE build4all_research.coupons
- coupon_id: integer PRIMARY KEY NOT NULL
- code: character varying NOT NULL
- discount_type: character varying NOT NULL
- discount_value: numeric NOT NULL
- minimum_order_amount: numeric NOT NULL
- maximum_discount_amount: numeric NULL
- valid_from: timestamp with time zone NULL
- valid_to: timestamp with time zone NULL
- maximum_uses: integer NULL
- active: boolean NOT NULL

TABLE build4all_research.customers
- customer_id: integer PRIMARY KEY NOT NULL
- customer_code: character varying NOT NULL
- city: character varying NULL
- region: character varying NULL
- created_at: timestamp with time zone NOT NULL

TABLE build4all_research.order_items
- order_item_id: integer PRIMARY KEY NOT NULL
- order_id: integer NOT NULL
- product_id: integer NOT NULL
- quantity: integer NOT NULL
- unit

In [7]:
# ============================================================
# CELL 6 — EXTRACT AND AUDIT THE FOUR RAG PDFs
# ============================================================

from pypdf import PdfReader
from pathlib import Path
import json
import shutil

pdf_files = sorted(
    Path("/content/drive/MyDrive/Build4All_Research/RAG/Extracted/documents").rglob("*.pdf")
)

raw_documents = []
document_audit = []

for document_id, pdf_path in enumerate(pdf_files):

    reader = PdfReader(str(pdf_path))
    page_texts = []

    for page_number, page in enumerate(
        reader.pages,
        start=1,
    ):
        extracted_text = (
            page.extract_text() or ""
        ).strip()

        page_texts.append(extracted_text)

        raw_documents.append({
            "document_id": document_id,
            "document_name": pdf_path.stem,
            "source_path": str(pdf_path),
            "page_number": page_number,
            "text": extracted_text,
        })

    full_text = "\n\n".join(page_texts)

    document_audit.append({
        "document_id": document_id,
        "document_name": pdf_path.stem,
        "page_count": len(reader.pages),
        "character_count": len(full_text),
        "empty_pages": sum(
            not text.strip()
            for text in page_texts
        ),
    })

    # Keep a working copy with a stable filename
    destination = (
        PATHS["documents"] / pdf_path.name
    )

    shutil.copy2(
        pdf_path,
        destination,
    )


raw_documents_path = (
    PATHS["documents"] / "extracted_pdf_pages.json"
)

raw_documents_path.write_text(
    json.dumps(
        raw_documents,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 70)
print("RAG PDF EXTRACTION AUDIT")
print("=" * 70)

for audit in document_audit:
    print(
        f"\nDocument: {audit['document_name']}"
    )
    print(
        f"Pages: {audit['page_count']}"
    )
    print(
        f"Characters: {audit['character_count']}"
    )
    print(
        f"Empty pages: {audit['empty_pages']}"
    )

print("\n" + "=" * 70)
print("TEXT PREVIEW")
print("=" * 70)

for document in raw_documents:
    preview = (
        document["text"][:250]
        .replace("\n", " ")
    )

    print(
        f"\n{document['document_name']} "
        f"— Page {document['page_number']}"
    )
    print(preview)

assert len(pdf_files) == 4
assert len(raw_documents) > 0

assert all(
    audit["empty_pages"] == 0
    for audit in document_audit
), "At least one PDF page has no extractable text."

assert all(
    audit["character_count"] > 500
    for audit in document_audit
), "At least one PDF contains insufficient text."

print("\nCELL 6 COMPLETED SUCCESSFULLY")
print("PDF files processed:", len(pdf_files))
print("Extracted pages:", len(raw_documents))
print("Extraction saved:", raw_documents_path)
print("RAG index built: False")
print("Qwen loaded: False")

RAG PDF EXTRACTION AUDIT

Document: Build4All_Customer_Regional_and_Shipping_Policy_v2.0
Pages: 2
Characters: 4155
Empty pages: 0

Document: Build4All_Product_Events_and_Conversion_Policy_v2.0
Pages: 2
Characters: 4119
Empty pages: 0

Document: Build4All_Product_Inventory_and_Coupon_Policy_v2.0
Pages: 2
Characters: 4287
Empty pages: 0

Document: Build4All_Sales_and_Order_Analytics_Policy_v2.0
Pages: 3
Characters: 6009
Empty pages: 0

TEXT PREVIEW

Build4All_Customer_Regional_and_Shipping_Policy_v2.0 — Page 1
Build4All Customer Regional and Shipping Policy Controlled knowledge source for multilingual Text to SQL retrieval Document ID B4A-DOC-CUSTOMER-002 Version 2.0 Authoritative schema build4all_research Status Controlled research source Scope Customers,

Build4All_Customer_Regional_and_Shipping_Policy_v2.0 — Page 2
Shipping method relationships An order references its shipping method through orders.shipping_method_id equals  shipping_methods.shipping_method_id. Use shipping_methods.na

In [8]:
# ============================================================
# CELL 7 — SENTENCE-SAFE SEMANTIC CHUNKING
# ============================================================

import re
import json

CHUNK_TARGET_CHARACTERS = 900
CHUNK_OVERLAP_CHARACTERS = 150


def clean_extracted_text(text):
    """
    Normalize PDF whitespace without changing its meaning.
    """

    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def split_into_sentences(text):
    """
    Split text at safe sentence or heading boundaries.
    Words are never cut in the middle.
    """

    text = clean_extracted_text(text)

    units = re.split(
        r"(?<=[.!?])\s+|\n+",
        text,
    )

    return [
        unit.strip()
        for unit in units
        if unit.strip()
    ]


def create_chunks(
    text,
    target_characters=900,
    overlap_characters=150,
):
    """
    Build sentence-safe chunks with limited semantic overlap.
    """

    sentences = split_into_sentences(text)
    chunks = []

    current_sentences = []
    current_length = 0

    for sentence in sentences:

        sentence_length = len(sentence) + 1

        if (
            current_sentences
            and current_length + sentence_length
            > target_characters
        ):
            chunk_text = " ".join(
                current_sentences
            ).strip()

            chunks.append(chunk_text)

            # Keep complete sentences for overlap
            overlap_sentences = []
            overlap_length = 0

            for previous_sentence in reversed(
                current_sentences
            ):
                if (
                    overlap_length
                    + len(previous_sentence)
                    > overlap_characters
                ):
                    break

                overlap_sentences.insert(
                    0,
                    previous_sentence,
                )

                overlap_length += (
                    len(previous_sentence) + 1
                )

            current_sentences = overlap_sentences
            current_length = sum(
                len(item) + 1
                for item in current_sentences
            )

        current_sentences.append(sentence)
        current_length += sentence_length

    if current_sentences:
        chunks.append(
            " ".join(current_sentences).strip()
        )

    return chunks


rag_chunks = []
global_chunk_id = 0

for document in raw_documents:

    page_chunks = create_chunks(
        document["text"],
        target_characters=(
            CHUNK_TARGET_CHARACTERS
        ),
        overlap_characters=(
            CHUNK_OVERLAP_CHARACTERS
        ),
    )

    for page_chunk_index, chunk_text in enumerate(
        page_chunks
    ):
        rag_chunks.append({
            "chunk_id": global_chunk_id,
            "document_id": document["document_id"],
            "document_name": document["document_name"],
            "page_number": document["page_number"],
            "page_chunk_index": page_chunk_index,
            "text": chunk_text,
            "character_count": len(chunk_text),
        })

        global_chunk_id += 1


chunks_path = (
    PATHS["documents"] / "rag_chunks.json"
)

chunks_path.write_text(
    json.dumps(
        rag_chunks,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 70)
print("RAG CHUNKING AUDIT")
print("=" * 70)

print("Source pages:", len(raw_documents))
print("Total chunks:", len(rag_chunks))
print(
    "Minimum chunk size:",
    min(chunk["character_count"] for chunk in rag_chunks),
)
print(
    "Maximum chunk size:",
    max(chunk["character_count"] for chunk in rag_chunks),
)
print(
    "Average chunk size:",
    round(
        sum(
            chunk["character_count"]
            for chunk in rag_chunks
        ) / len(rag_chunks),
        2,
    ),
)

print("\nCHUNKS BY DOCUMENT")

for document_name in sorted({
    chunk["document_name"]
    for chunk in rag_chunks
}):
    count = sum(
        chunk["document_name"] == document_name
        for chunk in rag_chunks
    )

    print(f"{document_name}: {count}")


print("\nFIRST CHUNK PREVIEW")
print("-" * 70)
print(rag_chunks[0]["text"][:700])

assert len(rag_chunks) >= len(raw_documents)
assert all(
    chunk["text"].strip()
    for chunk in rag_chunks
)
assert all(
    chunk["character_count"] <= 1200
    for chunk in rag_chunks
)

print("\nCELL 7 COMPLETED SUCCESSFULLY")
print("Chunks saved:", chunks_path)
print("RAG index built: False")
print("Qwen loaded: False")

RAG CHUNKING AUDIT
Source pages: 9
Total chunks: 26
Minimum chunk size: 341
Maximum chunk size: 899
Average chunk size: 783.15

CHUNKS BY DOCUMENT
Build4All_Customer_Regional_and_Shipping_Policy_v2.0: 6
Build4All_Product_Events_and_Conversion_Policy_v2.0: 6
Build4All_Product_Inventory_and_Coupon_Policy_v2.0: 6
Build4All_Sales_and_Order_Analytics_Policy_v2.0: 8

FIRST CHUNK PREVIEW
----------------------------------------------------------------------
Build4All Customer Regional and Shipping Policy Controlled knowledge source for multilingual Text to SQL retrieval Document ID B4A-DOC-CUSTOMER-002 Version 2.0 Authoritative schema build4all_research Status Controlled research source Scope Customers, regions, registrations, repeat purchasing, and shipping methods Purpose and authority This document defines customer, regional, registration, repeat-purchase, and shipping-method meanings for the frozen build4all_research PostgreSQL schema. It is authoritative for the controlled retrieval expe

In [9]:
# ============================================================
# CELL 8 — BUILD THE MULTILINGUAL FAISS RAG INDEX
# ============================================================

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import json
import time

EMBEDDING_MODEL_ID = (
    CONFIG["rag"]["embedding_model"]
)

print(
    "Loading embedding model:",
    EMBEDDING_MODEL_ID,
)

# Keep embeddings on CPU to preserve GPU memory for Qwen
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_ID,
    device="cpu",
)

passage_texts = [
    "passage: " + chunk["text"]
    for chunk in rag_chunks
]

embedding_start = time.perf_counter()

chunk_embeddings = embedding_model.encode(
    passage_texts,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

embedding_time = (
    time.perf_counter() - embedding_start
)

embedding_dimension = (
    chunk_embeddings.shape[1]
)

rag_index = faiss.IndexFlatIP(
    embedding_dimension
)

rag_index.add(chunk_embeddings)


# ------------------------------------------------------------
# Retrieval function
# ------------------------------------------------------------

def retrieve_chunks(question, top_k=None):

    if top_k is None:
        top_k = CONFIG["rag"]["top_k"]

    query_embedding = embedding_model.encode(
        ["query: " + question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")

    scores, indices = rag_index.search(
        query_embedding,
        top_k,
    )

    results = []

    for rank, (index_position, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1,
    ):
        chunk = rag_chunks[
            int(index_position)
        ].copy()

        chunk["rank"] = rank
        chunk["retrieval_score"] = float(score)

        results.append(chunk)

    return results


# ------------------------------------------------------------
# Save index and embeddings
# ------------------------------------------------------------

index_path = (
    PATHS["indexes"] / "build4all_rag.faiss"
)

embeddings_path = (
    PATHS["indexes"] / "chunk_embeddings.npy"
)

faiss.write_index(
    rag_index,
    str(index_path),
)

np.save(
    embeddings_path,
    chunk_embeddings,
)


# ------------------------------------------------------------
# Multilingual retrieval smoke tests
# ------------------------------------------------------------

retrieval_tests = [
    "How is recognized revenue calculated?",
    "كيف يتم حساب الإيرادات المعترف بها؟",
    "Comment calculer le taux de conversion vue-achat ?",
    "Kif mn7seb el active inventory value?",
]

print("=" * 70)
print("MULTILINGUAL RETRIEVAL SMOKE TEST")
print("=" * 70)

for question in retrieval_tests:

    results = retrieve_chunks(
        question,
        top_k=2,
    )

    print("\nQUESTION:", question)

    for result in results:
        print(
            f"Rank {result['rank']} | "
            f"Score {result['retrieval_score']:.4f} | "
            f"{result['document_name']} | "
            f"Page {result['page_number']}"
        )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert rag_index.ntotal == len(rag_chunks)
assert chunk_embeddings.shape[0] == len(rag_chunks)
assert np.isfinite(chunk_embeddings).all()
assert index_path.exists()
assert embeddings_path.exists()

print("\nCELL 8 COMPLETED SUCCESSFULLY")
print("Indexed chunks:", rag_index.ntotal)
print("Embedding dimension:", embedding_dimension)
print(
    "Embedding time:",
    round(embedding_time, 2),
    "seconds",
)
print("FAISS index saved:", index_path)
print("Embedding model device: CPU")
print("Qwen loaded: False")

Loading embedding model: intfloat/multilingual-e5-base


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

MULTILINGUAL RETRIEVAL SMOKE TEST

QUESTION: How is recognized revenue calculated?
Rank 1 | Score 0.8510 | Build4All_Sales_and_Order_Analytics_Policy_v2.0 | Page 1
Rank 2 | Score 0.8132 | Build4All_Sales_and_Order_Analytics_Policy_v2.0 | Page 2

QUESTION: كيف يتم حساب الإيرادات المعترف بها؟
Rank 1 | Score 0.7690 | Build4All_Sales_and_Order_Analytics_Policy_v2.0 | Page 1
Rank 2 | Score 0.7611 | Build4All_Sales_and_Order_Analytics_Policy_v2.0 | Page 3

QUESTION: Comment calculer le taux de conversion vue-achat ?
Rank 1 | Score 0.8591 | Build4All_Product_Events_and_Conversion_Policy_v2.0 | Page 1
Rank 2 | Score 0.8305 | Build4All_Product_Events_and_Conversion_Policy_v2.0 | Page 1

QUESTION: Kif mn7seb el active inventory value?
Rank 1 | Score 0.8274 | Build4All_Product_Inventory_and_Coupon_Policy_v2.0 | Page 1
Rank 2 | Score 0.8063 | Build4All_Product_Inventory_and_Coupon_Policy_v2.0 | Page 2

CELL 8 COMPLETED SUCCESSFULLY
Indexed chunks: 26
Embedding dimension: 768
Embedding time: 11.34 

In [10]:
# ============================================================
# CELL 9 — LOAD QWEN3-8B IN 4-BIT
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = CONFIG["model"]["model_id"]

# ------------------------------------------------------------
# 1. Configure 4-bit quantization
# ------------------------------------------------------------

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# ------------------------------------------------------------
# 2. Load tokenizer
# ------------------------------------------------------------

print("Loading Qwen tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# ------------------------------------------------------------
# 3. Load Qwen
# ------------------------------------------------------------

print("Loading Qwen3-8B in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    max_memory={
    0: "14GiB",
    "cpu": "8GiB",
},
)

model.eval()


assert model.training is False
print("Qwen3-8B loaded in 4-bit. GPU allocated:",
      round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")

Loading Qwen tokenizer...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-8B loaded in 4-bit. GPU allocated: 5.67 GB


In [11]:
# ============================================================
# CELL 10 — FROZEN QWEN AND RAG GENERATION FUNCTIONS
# ============================================================

import re
import time
import torch

# ------------------------------------------------------------
# 1. Build controlled vocabulary text
# ------------------------------------------------------------

CONTROLLED_VOCABULARY_TEXT = "\n".join(
    f"- {field}: {values}"
    for field, values
    in controlled_vocabulary.items()
)


# ------------------------------------------------------------
# 2. Build RAG context
# ------------------------------------------------------------

def build_rag_context(question):

    retrieved = retrieve_chunks(
        question,
        top_k=CONFIG["rag"]["top_k"],
    )

    context_parts = []
    current_length = 0

    for result in retrieved:

        chunk_block = (
            f"[Source: {result['document_name']}, "
            f"page {result['page_number']}, "
            f"score {result['retrieval_score']:.4f}]\n"
            f"{result['text']}"
        )

        if (
            current_length + len(chunk_block)
            > CONFIG["rag"]["max_context_characters"]
        ):
            break

        context_parts.append(chunk_block)
        current_length += len(chunk_block)

    return (
        "\n\n".join(context_parts),
        retrieved,
    )


# ------------------------------------------------------------
# 3. Extract SQL safely from Qwen response
# ------------------------------------------------------------

def extract_sql(raw_response):

    response = raw_response.strip()

    fenced_match = re.search(
        r"```(?:sql)?\s*(.*?)```",
        response,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if fenced_match:
        response = fenced_match.group(1).strip()

    sql_start = re.search(
        r"\b(SELECT|WITH)\b",
        response,
        flags=re.IGNORECASE,
    )

    if not sql_start:
        return ""

    response = response[
        sql_start.start():
    ].strip()

    if ";" in response:
        response = (
            response.split(";", 1)[0].strip()
            + ";"
        )

    return response


# ------------------------------------------------------------
# 4. Frozen Text-to-SQL generation function
# ------------------------------------------------------------

def generate_sql(
    question,
    use_rag=False,
):

    rag_context = ""
    retrieved_chunks = []

    if use_rag:
        rag_context, retrieved_chunks = (
            build_rag_context(question)
        )

    system_prompt = """
You are a multilingual PostgreSQL Text-to-SQL system.

Generate exactly one read-only PostgreSQL query that answers
the user's business question.

Mandatory rules:
1. Use only the provided database schema and columns.
2. Use the build4all_research schema explicitly.
3. Generate only SELECT or WITH queries.
4. Never generate INSERT, UPDATE, DELETE, DROP, ALTER,
   CREATE, TRUNCATE, GRANT, REVOKE, COPY, or CALL.
5. Follow the controlled vocabulary exactly.
6. Do not invent tables, columns, filters, joins, or values.
7. Return SQL only, with no explanation or Markdown.
""".strip()

    user_parts = [
        "DATABASE SCHEMA:",
        DATABASE_SCHEMA_TEXT,
        "",
        "CONTROLLED DATABASE VALUES:",
        CONTROLLED_VOCABULARY_TEXT,
    ]

    if use_rag:
        user_parts.extend([
            "",
            "RETRIEVED BUSINESS POLICY:",
            rag_context,
            "",
            "Use the retrieved policy only when it is relevant.",
        ])

    user_parts.extend([
        "",
        "BUSINESS QUESTION:",
        question,
        "",
        "POSTGRESQL QUERY:",
    ])

    user_prompt = "\n".join(user_parts)

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(model.device)

    start_time = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=(
                CONFIG["model"]["max_new_tokens"]
            ),
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generation_time = (
        time.perf_counter() - start_time
    )

    new_tokens = generated_ids[
        :,
        model_inputs["input_ids"].shape[1]:,
    ]

    raw_response = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True,
    ).strip()

    generated_sql = extract_sql(
        raw_response
    )

    return {
        "question": question,
        "system": (
            "qwen_with_rag"
            if use_rag
            else "qwen_alone"
        ),
        "sql": generated_sql,
        "raw_response": raw_response,
        "latency_seconds": round(
            generation_time,
            4,
        ),
        "retrieved_chunks": retrieved_chunks,
        "rag_context_characters": len(
            rag_context
        ),
    }


print("=" * 70)
print("FROZEN GENERATION PIPELINES")
print("=" * 70)
print("Qwen Alone ready: True")
print("Qwen + RAG ready: True")
print("Same model: True")
print("Same database schema: True")
print("Same generation settings: True")
print("Only difference: retrieved business policy")
print("Generations performed: 0")

print("\nCELL 10 COMPLETED SUCCESSFULLY")

FROZEN GENERATION PIPELINES
Qwen Alone ready: True
Qwen + RAG ready: True
Same model: True
Same database schema: True
Same generation settings: True
Only difference: retrieved business policy
Generations performed: 0

CELL 10 COMPLETED SUCCESSFULLY


In [12]:
# ============================================================
# CELL 11 — SQL SAFETY, EXECUTION AND RESULT NORMALIZATION
# ============================================================

from decimal import Decimal
from datetime import date, datetime
import sqlglot
from sqlglot import expressions as exp


ALLOWED_TABLES = set(discovered_tables)

FORBIDDEN_EXPRESSIONS = (
    exp.Insert,
    exp.Update,
    exp.Delete,
    exp.Drop,
    exp.Create,
    exp.Alter,
    exp.Command,
    exp.Merge,
    exp.TruncateTable,
)


# ------------------------------------------------------------
# 1. Validate generated SQL
# ------------------------------------------------------------

def validate_readonly_sql(sql_text):

    if not sql_text or not sql_text.strip():
        return {
            "safe": False,
            "reason": "Empty SQL",
            "parsed": None,
        }

    try:
        statements = [
            statement
            for statement in sqlglot.parse(
                sql_text,
                read="postgres",
            )
            if statement is not None
        ]

    except Exception as error:
        return {
            "safe": False,
            "reason": f"SQL parse error: {error}",
            "parsed": None,
        }

    if len(statements) != 1:
        return {
            "safe": False,
            "reason": "Exactly one SQL statement is required",
            "parsed": None,
        }

    parsed = statements[0]

    if not isinstance(
        parsed,
        (
            exp.Select,
            exp.Union,
            exp.Intersect,
            exp.Except,
        ),
    ):
        return {
            "safe": False,
            "reason": (
                "Only SELECT or WITH queries are allowed"
            ),
            "parsed": parsed,
        }

    for forbidden_type in FORBIDDEN_EXPRESSIONS:
        if parsed.find(forbidden_type):
            return {
                "safe": False,
                "reason": (
                    f"Forbidden SQL operation: "
                    f"{forbidden_type.__name__}"
                ),
                "parsed": parsed,
            }

    referenced_tables = []

    for table in parsed.find_all(exp.Table):

        table_name = table.name
        table_schema = table.db

        referenced_tables.append({
            "schema": table_schema,
            "table": table_name,
        })

        if table_name not in ALLOWED_TABLES:
            return {
                "safe": False,
                "reason": (
                    f"Unknown or forbidden table: "
                    f"{table_name}"
                ),
                "parsed": parsed,
            }

        if (
            table_schema
            and table_schema != SCHEMA_NAME
        ):
            return {
                "safe": False,
                "reason": (
                    f"Forbidden schema: "
                    f"{table_schema}"
                ),
                "parsed": parsed,
            }

    if not referenced_tables:
        return {
            "safe": False,
            "reason": "No approved database table referenced",
            "parsed": parsed,
        }

    return {
        "safe": True,
        "reason": "Read-only SQL validated",
        "parsed": parsed,
        "referenced_tables": referenced_tables,
    }


# ------------------------------------------------------------
# 2. Normalize database values
# ------------------------------------------------------------

def normalize_value(value):

    if isinstance(value, Decimal):
        return round(float(value), 6)

    if isinstance(value, (datetime, date)):
        return value.isoformat()

    if isinstance(value, float):
        return round(value, 6)

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace",
        )

    return value


def normalize_result(columns, rows):

    normalized_rows = [
        tuple(
            normalize_value(value)
            for value in row
        )
        for row in rows
    ]

    # Result comparison should not fail because of row order
    sorted_rows = sorted(
        normalized_rows,
        key=lambda row: repr(row),
    )

    return {
        "columns": [
            str(column).lower()
            for column in columns
        ],
        "rows": sorted_rows,
    }


# ------------------------------------------------------------
# 3. Execute only validated SQL
# ------------------------------------------------------------

def execute_safe_sql(sql_text):

    validation = validate_readonly_sql(
        sql_text
    )

    if not validation["safe"]:
        return {
            "executed": False,
            "safe": False,
            "error": validation["reason"],
            "columns": [],
            "rows": [],
            "normalized_result": None,
        }

    try:
        with get_db_connection() as connection:

            with connection.cursor() as cursor:

                cursor.execute(sql_text)

                columns = [
                    description.name
                    for description in cursor.description
                ]

                rows = cursor.fetchall()

        return {
            "executed": True,
            "safe": True,
            "error": None,
            "columns": columns,
            "rows": rows,
            "normalized_result": normalize_result(
                columns,
                rows,
            ),
        }

    except Exception as error:
        return {
            "executed": False,
            "safe": True,
            "error": str(error),
            "columns": [],
            "rows": [],
            "normalized_result": None,
        }


# ------------------------------------------------------------
# 4. Safety smoke tests
# ------------------------------------------------------------

safe_test_sql = """
SELECT status, COUNT(*) AS order_count
FROM build4all_research.orders
GROUP BY status
ORDER BY status;
"""

unsafe_test_sql = """
DELETE FROM build4all_research.orders;
"""

unknown_table_test_sql = """
SELECT *
FROM private_customer_passwords;
"""

safe_validation = validate_readonly_sql(
    safe_test_sql
)

unsafe_validation = validate_readonly_sql(
    unsafe_test_sql
)

unknown_validation = validate_readonly_sql(
    unknown_table_test_sql
)

safe_execution = execute_safe_sql(
    safe_test_sql
)


print("=" * 70)
print("SQL SAFETY AND EXECUTION AUDIT")
print("=" * 70)

print(
    "Safe SELECT accepted:",
    safe_validation["safe"],
)

print(
    "DELETE rejected:",
    not unsafe_validation["safe"],
)

print(
    "Unknown table rejected:",
    not unknown_validation["safe"],
)

print(
    "Safe SELECT executed:",
    safe_execution["executed"],
)

print(
    "Returned rows:",
    len(safe_execution["rows"]),
)

print(
    "Execution error:",
    safe_execution["error"],
)


assert safe_validation["safe"] is True
assert unsafe_validation["safe"] is False
assert unknown_validation["safe"] is False
assert safe_execution["executed"] is True

print("\nCELL 11 COMPLETED SUCCESSFULLY")
print("SQL validator ready: True")
print("Read-only execution ready: True")
print("Qwen generations performed: 0")

SQL SAFETY AND EXECUTION AUDIT
Safe SELECT accepted: True
DELETE rejected: True
Unknown table rejected: True
Safe SELECT executed: True
Returned rows: 6
Execution error: None

CELL 11 COMPLETED SUCCESSFULLY
SQL validator ready: True
Read-only execution ready: True
Qwen generations performed: 0


In [13]:
# ============================================================
# CELL 14 — UPGRADE VALIDATOR FOR SAFE WITH / CTE QUERIES
# ============================================================

def validate_readonly_sql(sql_text):

    if not sql_text or not sql_text.strip():
        return {
            "safe": False,
            "reason": "Empty SQL",
            "parsed": None,
        }

    try:
        statements = [
            statement
            for statement in sqlglot.parse(
                sql_text,
                read="postgres",
            )
            if statement is not None
        ]

    except Exception as error:
        return {
            "safe": False,
            "reason": f"SQL parse error: {error}",
            "parsed": None,
        }

    if len(statements) != 1:
        return {
            "safe": False,
            "reason": (
                "Exactly one SQL statement is required"
            ),
            "parsed": None,
        }

    parsed = statements[0]

    if not isinstance(
        parsed,
        (
            exp.Select,
            exp.Union,
            exp.Intersect,
            exp.Except,
        ),
    ):
        return {
            "safe": False,
            "reason": (
                "Only SELECT or WITH queries are allowed"
            ),
            "parsed": parsed,
        }

    for forbidden_type in FORBIDDEN_EXPRESSIONS:
        if parsed.find(forbidden_type):
            return {
                "safe": False,
                "reason": (
                    f"Forbidden SQL operation: "
                    f"{forbidden_type.__name__}"
                ),
                "parsed": parsed,
            }

    # Collect temporary table names defined by WITH clauses
    cte_names = {
        cte.alias_or_name
        for cte in parsed.find_all(exp.CTE)
        if cte.alias_or_name
    }

    referenced_tables = []

    for table in parsed.find_all(exp.Table):

        table_name = table.name
        table_schema = table.db

        # CTE names are valid temporary query relations
        if (
            table_name in cte_names
            and not table_schema
        ):
            continue

        referenced_tables.append({
            "schema": table_schema,
            "table": table_name,
        })

        if table_name not in ALLOWED_TABLES:
            return {
                "safe": False,
                "reason": (
                    f"Unknown or forbidden table: "
                    f"{table_name}"
                ),
                "parsed": parsed,
            }

        if (
            table_schema
            and table_schema != SCHEMA_NAME
        ):
            return {
                "safe": False,
                "reason": (
                    f"Forbidden schema: "
                    f"{table_schema}"
                ),
                "parsed": parsed,
            }

    if not referenced_tables:
        return {
            "safe": False,
            "reason": (
                "No approved database table referenced"
            ),
            "parsed": parsed,
        }

    return {
        "safe": True,
        "reason": "Read-only SQL validated",
        "parsed": parsed,
        "referenced_tables": referenced_tables,
        "cte_names": sorted(cte_names),
    }


# ------------------------------------------------------------
# Test a real read-only CTE
# ------------------------------------------------------------

cte_test_sql = """
WITH recognized_customer_orders AS (
    SELECT
        customer_id,
        COUNT(*) AS recognized_order_count
    FROM build4all_research.orders
    WHERE status IN ('completed', 'shipped')
    GROUP BY customer_id
)
SELECT COUNT(*) AS repeat_customer_count
FROM recognized_customer_orders
WHERE recognized_order_count > 1;
"""

cte_validation = validate_readonly_sql(
    cte_test_sql
)

cte_execution = execute_safe_sql(
    cte_test_sql
)

print("=" * 70)
print("CTE SAFETY VALIDATION")
print("=" * 70)

print("CTE safe:", cte_validation["safe"])
print(
    "Detected CTE names:",
    cte_validation.get("cte_names"),
)
print("CTE executed:", cte_execution["executed"])
print("Execution error:", cte_execution["error"])
print("Result:", cte_execution["rows"])

assert cte_validation["safe"] is True
assert cte_execution["executed"] is True
assert "recognized_customer_orders" in (
    cte_validation["cte_names"]
)

print("\nCELL 14 COMPLETED SUCCESSFULLY")
print("SELECT validation ready: True")
print("WITH validation ready: True")
print("Final benchmark started: False")

CTE SAFETY VALIDATION
CTE safe: True
Detected CTE names: ['recognized_customer_orders']
CTE executed: True
Execution error: None
Result: [(393,)]

CELL 14 COMPLETED SUCCESSFULLY
SELECT validation ready: True
WITH validation ready: True
Final benchmark started: False


In [14]:
# ================================================================
# CELL 18 — EXECUTION ACCURACY EVALUATION FRAMEWORK
# ================================================================

from decimal import Decimal
from datetime import date, datetime
import json
import math
import os
import pandas as pd


# ------------------------------------------------
# 1. Normalize database values
# ------------------------------------------------
def normalize_evaluation_value(value):
    if value is None:
        return None

    if isinstance(value, bool):
        return value

    if isinstance(value, Decimal):
        if value == value.to_integral():
            return str(value.quantize(Decimal("1")))

        normalized = format(value.normalize(), "f")
        return normalized.rstrip("0").rstrip(".") if "." in normalized else normalized

    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"

        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"

        return format(value, ".10g")

    if isinstance(value, (datetime, date)):
        return value.isoformat()

    if isinstance(value, (int, str)):
        return str(value)

    return str(value)


def normalize_evaluation_rows(rows):
    normalized_rows = []

    for row in rows:
        normalized_row = tuple(
            normalize_evaluation_value(value)
            for value in row
        )
        normalized_rows.append(normalized_row)

    # Row order should not affect execution accuracy.
    return sorted(
        normalized_rows,
        key=lambda row: json.dumps(
            row,
            ensure_ascii=False,
            sort_keys=True,
            default=str
        )
    )


# ------------------------------------------------
# 2. Execute evaluated SQL safely
# ------------------------------------------------
def execute_for_evaluation(sql):
    validation = validate_readonly_sql(sql)

    if not validation["safe"]:
        return {
            "executed": False,
            "safe": False,
            "validation_reason": validation.get("reason"),
            "columns": [],
            "rows": [],
            "error": None
        }

    connection = None

    try:
        connection = get_db_connection()

        with connection.cursor() as cursor:
            cursor.execute("SET statement_timeout TO 30000")
            cursor.execute(sql)

            columns = [
                description.name
                for description in cursor.description
            ]

            rows = cursor.fetchall()

        connection.rollback()

        return {
            "executed": True,
            "safe": True,
            "validation_reason": validation.get("reason"),
            "columns": columns,
            "rows": rows,
            "error": None
        }

    except Exception as error:
        if connection is not None:
            connection.rollback()

        return {
            "executed": False,
            "safe": True,
            "validation_reason": validation.get("reason"),
            "columns": [],
            "rows": [],
            "error": f"{type(error).__name__}: {error}"
        }

    finally:
        if connection is not None:
            connection.close()


# ------------------------------------------------
# 3. Compare generated result with gold result
# ------------------------------------------------
def evaluate_generated_sql(generated_sql, gold_rows):
    execution = execute_for_evaluation(generated_sql)

    predicted_rows = normalize_evaluation_rows(
        execution["rows"]
    )

    expected_rows = normalize_evaluation_rows(
        gold_rows
    )

    same_column_count = True

    if predicted_rows and expected_rows:
        same_column_count = (
            len(predicted_rows[0]) ==
            len(expected_rows[0])
        )

    execution_correct = (
        execution["executed"]
        and same_column_count
        and predicted_rows == expected_rows
    )

    return {
        "execution_correct": execution_correct,
        "sql_safe": execution["safe"],
        "sql_executed": execution["executed"],
        "execution_error": execution["error"],
        "predicted_row_count": len(predicted_rows),
        "gold_row_count": len(expected_rows),
        "predicted_rows": predicted_rows,
        "gold_rows": expected_rows
    }




# Strict experiment used this version (search_path enabled):
def execute_for_evaluation(sql):
    validation = validate_readonly_sql(sql)

    if not validation["safe"]:
        return {
            "executed": False,
            "safe": False,
            "validation_reason": validation.get("reason"),
            "columns": [],
            "rows": [],
            "error": None
        }

    connection = None

    try:
        connection = get_db_connection()

        with connection.cursor() as cursor:
            cursor.execute(
                "SET statement_timeout TO 30000"
            )

            # Allow both:
            # FROM orders
            # FROM build4all_research.orders
            cursor.execute(
                "SET search_path TO "
                "build4all_research, public"
            )

            cursor.execute(sql)

            columns = [
                description.name
                for description in cursor.description
            ]

            rows = cursor.fetchall()

        connection.rollback()

        return {
            "executed": True,
            "safe": True,
            "validation_reason": validation.get("reason"),
            "columns": columns,
            "rows": rows,
            "error": None
        }

    except Exception as error:
        if connection is not None:
            connection.rollback()

        return {
            "executed": False,
            "safe": True,
            "validation_reason": validation.get("reason"),
            "columns": [],
            "rows": [],
            "error": f"{type(error).__name__}: {error}"
        }

    finally:
        if connection is not None:
            connection.close()



print("Evaluation functions ready")

Evaluation functions ready


In [15]:
# Fine-tuning prompt used in training and in the strict evaluations
FINE_TUNING_SYSTEM_PROMPT = """
You are a multilingual PostgreSQL Text-to-SQL system.

Generate exactly one read-only PostgreSQL query that answers
the user's business question.

Mandatory rules:
1. Use only the provided database schema and columns.
2. Use the build4all_research schema explicitly.
3. Generate only SELECT or WITH queries.
4. Never generate INSERT, UPDATE, DELETE, DROP, ALTER,
   CREATE, TRUNCATE, GRANT, REVOKE, COPY, or CALL.
5. Follow the controlled vocabulary exactly.
6. Do not invent tables, columns, filters, joins, or values.
7. Return SQL only, with no explanation or Markdown.
""".strip()

# ------------------------------------------------------------
# 3. Build the same prompt used for generation
# ------------------------------------------------------------

def build_fine_tuning_user_prompt(question):

    return "\n".join([
        "DATABASE SCHEMA:",
        DATABASE_SCHEMA_TEXT,
        "",
        "CONTROLLED DATABASE VALUES:",
        CONTROLLED_VOCABULARY_TEXT,
        "",
        "BUSINESS QUESTION:",
        str(question).strip(),
        "",
        "POSTGRESQL QUERY:",
    ])


print("Fine-tuning prompt ready")

Fine-tuning prompt ready


In [16]:
# Strict template-family split — exact original code
import os, pandas as pd
custom_dataset_df = pd.read_csv(DATASET_CSV)
RESULTS_DIRECTORY = str(REV / "build4all_final_experiment" / "results")
os.makedirs(RESULTS_DIRECTORY, exist_ok=True)

# ============================================================
# STRICT TEMPLATE-FAMILY SPLIT
# No SQL template may cross train/validation/final-test
# ============================================================

import os
import random
import pandas as pd

STRICT_SPLIT_SEED = 42
random.seed(STRICT_SPLIT_SEED)

# One row per unique intent
intent_level_df = (
    custom_dataset_df[
        [
            "intent_id",
            "template_family",
            "domain",
            "complexity",
        ]
    ]
    .drop_duplicates(subset=["intent_id"])
    .reset_index(drop=True)
)

family_sizes = (
    intent_level_df
    .groupby("template_family")["intent_id"]
    .nunique()
    .to_dict()
)

families = sorted(family_sizes.keys())

# ------------------------------------------------------------
# Search for a family-disjoint split close to:
# 100 train intents, 25 validation intents, 25 final intents
# ------------------------------------------------------------

best_assignment = None
best_score = float("inf")

for attempt in range(50000):

    shuffled_families = families.copy()
    random.shuffle(shuffled_families)

    cumulative_sizes = []
    running_total = 0

    for family in shuffled_families:
        running_total += family_sizes[family]
        cumulative_sizes.append(running_total)

    validation_end = min(
        range(1, len(shuffled_families) - 1),
        key=lambda index: abs(
            cumulative_sizes[index - 1] - 25
        ),
    )

    validation_families = set(
        shuffled_families[:validation_end]
    )

    remaining_families = (
        shuffled_families[validation_end:]
    )

    remaining_cumulative = []
    running_total = 0

    for family in remaining_families:
        running_total += family_sizes[family]
        remaining_cumulative.append(running_total)

    final_end = min(
        range(1, len(remaining_families)),
        key=lambda index: abs(
            remaining_cumulative[index - 1] - 25
        ),
    )

    final_families = set(
        remaining_families[:final_end]
    )

    train_families = (
        set(families)
        - validation_families
        - final_families
    )

    validation_intent_count = sum(
        family_sizes[family]
        for family in validation_families
    )

    final_intent_count = sum(
        family_sizes[family]
        for family in final_families
    )

    train_intent_count = sum(
        family_sizes[family]
        for family in train_families
    )

    score = (
        abs(train_intent_count - 100)
        + abs(validation_intent_count - 25)
        + abs(final_intent_count - 25)
    )

    if score < best_score:
        best_score = score
        best_assignment = {
            "train": train_families,
            "validation": validation_families,
            "final_test": final_families,
        }

    if score == 0:
        break


# ------------------------------------------------------------
# Apply strict split to all four translations
# ------------------------------------------------------------

def assign_strict_split(template_family):

    if template_family in best_assignment["train"]:
        return "train"

    if template_family in best_assignment["validation"]:
        return "validation"

    if template_family in best_assignment["final_test"]:
        return "final_test"

    raise ValueError(
        f"Unassigned template family: {template_family}"
    )


strict_dataset_df = custom_dataset_df.copy()

# Keep old split for audit only
strict_dataset_df["original_split"] = (
    strict_dataset_df["split"]
)

strict_dataset_df["split"] = (
    strict_dataset_df["template_family"]
    .map(assign_strict_split)
)

strict_train_df = strict_dataset_df[
    strict_dataset_df["split"] == "train"
].copy().reset_index(drop=True)

strict_validation_df = strict_dataset_df[
    strict_dataset_df["split"] == "validation"
].copy().reset_index(drop=True)

strict_final_test_df = strict_dataset_df[
    strict_dataset_df["split"] == "final_test"
].copy().reset_index(drop=True)


# ------------------------------------------------------------
# Leakage audits
# ------------------------------------------------------------

train_families = set(
    strict_train_df["template_family"]
)

validation_families = set(
    strict_validation_df["template_family"]
)

final_families = set(
    strict_final_test_df["template_family"]
)

assert train_families.isdisjoint(validation_families)
assert train_families.isdisjoint(final_families)
assert validation_families.isdisjoint(final_families)

intent_split_counts = (
    strict_dataset_df
    .groupby("intent_id")["split"]
    .nunique()
)

assert int(intent_split_counts.max()) == 1

translation_counts = (
    strict_dataset_df
    .groupby("intent_id")["language"]
    .nunique()
)

assert int(translation_counts.min()) == 4
assert int(translation_counts.max()) == 4


# ------------------------------------------------------------
# Save revised dataset
# ------------------------------------------------------------

strict_dataset_path = os.path.join(
    RESULTS_DIRECTORY,
    "build4all_custom600_strict_template_split.csv",
)

strict_dataset_df.to_csv(
    strict_dataset_path,
    index=False,
)


# ------------------------------------------------------------
# Display audit
# ------------------------------------------------------------

strict_summary = (
    strict_dataset_df
    .groupby("split")
    .agg(
        questions=("question_id", "count"),
        intents=("intent_id", "nunique"),
        template_families=(
            "template_family",
            "nunique",
        ),
        languages=("language", "nunique"),
    )
    .reset_index()
)

print("=" * 70)
print("STRICT TEMPLATE-FAMILY SPLIT")
print("=" * 70)

display(strict_summary)

print(
    "\nTemplate families crossing splits:",
    0,
)

print(
    "Intent translations crossing splits:",
    0,
)

print(
    "Train/validation template overlap:",
    len(train_families & validation_families),
)

print(
    "Train/final-test template overlap:",
    len(train_families & final_families),
)

print(
    "Validation/final-test template overlap:",
    len(validation_families & final_families),
)

print("\nSTRICT VALIDATION FAMILIES")

for family in sorted(validation_families):
    print("-", family)

print("\nSTRICT FINAL-TEST FAMILIES — LOCKED")

for family in sorted(final_families):
    print("-", family)

print("\nCurrent trained adapter valid for this new split: False")
print("Reason: it already saw some newly held-out templates.")
print("Strict dataset saved:", strict_dataset_path)

EXPECTED_VALIDATION = {"orders_average_tax_by_status","payment_method_count","payment_method_paid_total",
    "products_average_price_by_status","products_categories_by_status","products_inventory_value_by_status"}
EXPECTED_TEST = {"events_count_by_type","events_top_products_by_type","orders_average_amount_by_status",
    "payments_orders_by_status","products_count_by_status"}
assert set(strict_validation_df.template_family) == EXPECTED_VALIDATION
assert set(strict_final_test_df.template_family) == EXPECTED_TEST
print("\nSplit identical to the paper: True")
print("(Ignore the line 'Current trained adapter valid ... False': it refers to an older preliminary adapter. The strict adapter loaded below was trained on exactly this split.)")

STRICT TEMPLATE-FAMILY SPLIT


,split,questions,intents,template_families,languages
0,final_test,100,25,5,4
1,train,400,100,20,4
2,validation,100,25,6,4



Template families crossing splits: 0
Intent translations crossing splits: 0
Train/validation template overlap: 0
Train/final-test template overlap: 0
Validation/final-test template overlap: 0

STRICT VALIDATION FAMILIES
- orders_average_tax_by_status
- payment_method_count
- payment_method_paid_total
- products_average_price_by_status
- products_categories_by_status
- products_inventory_value_by_status

STRICT FINAL-TEST FAMILIES — LOCKED
- events_count_by_type
- events_top_products_by_type
- orders_average_amount_by_status
- payments_orders_by_status
- products_count_by_status

Current trained adapter valid for this new split: False
Reason: it already saw some newly held-out templates.
Strict dataset saved: /content/drive/MyDrive/Build4All_Research/Revision_Experiments/build4all_final_experiment/results/build4all_custom600_strict_template_split.csv

Split identical to the paper: True
(Ignore the line 'Current trained adapter valid ... False': it refers to an older preliminary adapter

In [17]:
# A-final — Attach the frozen strict QLoRA adapter (used by every experiment;
# "original Qwen" runs switch it off with model.disable_adapter()).
from peft import PeftModel
if not hasattr(model, "peft_config"):
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
model.eval()
print("Strict adapter attached:", hasattr(model, "peft_config"))
print("PART A COMPLETE")

Strict adapter attached: True
PART A COMPLETE


In [18]:
# B0 — Shared experiment runner (run this once after Part A)
import json, time, contextlib
import pandas as pd

_gold_cache = {}

def gold_rows_for(gold_sql):
    if gold_sql not in _gold_cache:
        g = execute_for_evaluation(gold_sql)
        if not g["executed"]:
            raise RuntimeError("Gold SQL failed: " + str(g["error"]))
        _gold_cache[gold_sql] = g["rows"]
    return _gold_cache[gold_sql]


def run_experiment(name, frame, generate_fn, adapter=True):
    """Generate + execute + score every row of `frame`. Resumable."""
    out_jsonl = REV / f"{name}.jsonl"
    done = {}
    if out_jsonl.exists():
        for line in out_jsonl.read_text(encoding="utf-8").splitlines():
            try:
                r = json.loads(line); done[r["question_id"]] = r
            except Exception:
                pass
    todo = frame[~frame["question_id"].isin(done)]
    print(f"{name}: {len(done)} already done, {len(todo)} to run")

    ctx = contextlib.nullcontext() if adapter else model.disable_adapter()
    with ctx:
        for k, (_, row) in enumerate(todo.iterrows(), start=1):
            try:
                gen = generate_fn(row)
                ev = evaluate_generated_sql(gen["sql"], gold_rows_for(row["gold_sql"]))
                rec = dict(sql=gen["sql"], latency_seconds=gen.get("latency_seconds"),
                           sql_safe=bool(ev["sql_safe"]), sql_executed=bool(ev["sql_executed"]),
                           execution_correct=bool(ev["execution_correct"]),
                           execution_error=ev["execution_error"],
                           extra=gen.get("extra"))
            except Exception as error:
                rec = dict(sql="", latency_seconds=None, sql_safe=False, sql_executed=False,
                           execution_correct=False, execution_error=f"{type(error).__name__}: {error}",
                           extra=None)
            rec.update(question_id=row["question_id"], intent_id=row["intent_id"],
                       template_family=row["template_family"], language=row["language"],
                       split=row["split"], question=row["question"], gold_sql=row["gold_sql"],
                       experiment=name)
            with out_jsonl.open("a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
            done[row["question_id"]] = rec
            print(f"[{len(done):03d}/{len(frame)}] {row['question_id']} correct={rec['execution_correct']}")

    res = pd.DataFrame([done[q] for q in frame["question_id"] if q in done])
    res.to_csv(REV / f"{name}.csv", index=False)
    print("\n" + "=" * 60 + f"\n{name}\n" + "=" * 60)
    print(res.groupby("split")["execution_correct"].agg(["sum", "count", "mean"]))
    print(res.groupby(["split", "language"])["execution_correct"].sum().unstack())
    print("Saved:", REV / f"{name}.csv")
    return res


heldout_df = pd.concat([strict_validation_df, strict_final_test_df]).reset_index(drop=True)
assert len(heldout_df) == 200
print("Runner ready. Held-out questions:", len(heldout_df))

Runner ready. Held-out questions: 200


## Part D — Policy-dependent fresh test

In [19]:
# D0 — Load the PD-Test and audit every gold query (must all pass before generating)
import pandas as pd
pd_candidates = sorted(ROOT.rglob("build4all_policy_test_v1.csv"))
assert pd_candidates, "Upload build4all_policy_test_v1.csv to Revision_Experiments/"
pd_df = pd.read_csv(pd_candidates[0])
assert len(pd_df) == 128 and pd_df["intent_id"].nunique() == 32
assert set(pd_df["template_family"]).isdisjoint(set(custom_dataset_df["template_family"]))

audit = []
for intent, g in pd_df.groupby("intent_id"):
    r = execute_for_evaluation(g["gold_sql"].iloc[0])
    audit.append(dict(intent_id=intent, family=g["template_family"].iloc[0],
                      type=g["question_type"].iloc[0], executed=r["executed"],
                      rows=len(r["rows"]), first_row=str(r["rows"][:1])[:80], error=r["error"]))
audit_df = pd.DataFrame(audit)
audit_df.to_csv(REV / "D0_pd_gold_audit.csv", index=False)
print(audit_df.to_string())
assert audit_df["executed"].all(), "A gold query failed — send this table to Claude."
print("\nAll 32 gold queries executed. Rows with 0 results:", int((audit_df.rows == 0).sum()))

     intent_id                          family              type  executed  rows                                        first_row error
0   B4A-PD-E01       total_shipping_all_orders  explicit_control      True     1                         [(Decimal('54047.00'),)]  None
1   B4A-PD-E02       average_subtotal_refunded  explicit_control      True     1                          [(Decimal('3389.60'),)]  None
2   B4A-PD-E03          customers_created_2026  explicit_control      True     1                                         [(132,)]  None
3   B4A-PD-E04          max_price_discontinued  explicit_control      True     1                           [(Decimal('672.59'),)]  None
4   B4A-PD-E05  tax_rules_applying_to_shipping  explicit_control      True     1                                           [(5,)]  None
5   B4A-PD-E06                 root_categories  explicit_control      True     1                                           [(4,)]  None
6   B4A-PD-E07   customers_with_pending_orders  

### Pre-registered evaluation protocol (fixed on 6 Oct 2026, before any PD-Test generation)

**Systems compared on the 128 PD-Test questions:** fine-tuned direct; fine-tuned + always-on RAG;
controller v1 (frozen, tau = 0.93); controller v2a (retrieval-feature logistic regression, tau chosen by
leave-one-family-out on the 200 old held-out questions); controller v2b (policy-term similarity rule,
tau chosen on the same 200 questions); oracle (upper bound). Controllers only *select* between the two
generated answers, so no extra generation is needed for them.

**A controller is called useful if both hold:**
1. its accuracy is no more than 2 questions below the better of direct and always-on RAG, **and** it calls RAG
   on fewer questions than always-on RAG;
2. it routes policy-dependent questions to RAG at a clearly higher rate than control questions.

All results will be reported whatever they are. Nothing (prompts, thresholds, features, data) changes after D2/D3 start.
Run **D0 → D1 (freeze) → D2 → D3 → (D4) → D5**.

In [20]:
CTRL_SRC = '# ============================================================\n# FROZEN CONTROLLER DEFINITIONS (pre-registered before any PD-Test result exists)\n# Training data: the 200 old held-out questions (validation + locked test of the\n# strict benchmark) with the fine-tuned direct / RAG outcomes. No PD-Test labels.\n# ============================================================\nimport numpy as np, pandas as pd\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\n\nRETRIEVAL_FEATURES = ["score_1", "score_2", "score_3", "score_4", "score_5", "score_6",\n                      "top_score", "mean_score", "std_score", "gap_1_2", "question_length"]\n\n# Business metrics defined in the four policy documents (names only, copied from the PDFs).\nPOLICY_METRIC_NAMES = [\n    "recognized order revenue", "gross merchandise value", "net merchandise sales",\n    "average order value", "order count by status", "operational backlog", "cancellation rate",\n    "refunded order count", "daily order volume", "recognized coupon discount",\n    "recognized product sales by category", "recognized shipping amount", "effective tax rate",\n    "collected paid amount", "pending payment amount", "failed payment rate",\n    "refunded payment amount", "paid payment mismatch", "product count by catalog status",\n    "active products with fewer than ten units", "active inventory value", "current sale price",\n    "coupon validity", "event count by type", "view-to-purchase conversion",\n    "purchasing customers by region", "monthly customer registrations",\n    "repeat purchasing customers", "order count by shipping method",\n]\n\n\ndef select_threshold(scores, direct_ok, rag_ok, grid):\n    """Pick the threshold with the highest accuracy; among ties, the MIDDLE of the\n    widest tied run (robust to small score shifts, unlike the extreme of a plateau)."""\n    acc = np.array([np.where(scores >= t, rag_ok, direct_ok).mean() for t in grid])\n    best = acc.max()\n    runs, cur = [], []\n    for t, a in zip(grid, acc):\n        if a == best:\n            cur.append(t)\n        elif cur:\n            runs.append(cur); cur = []\n    if cur:\n        runs.append(cur)\n    widest = max(runs, key=len)\n    return float(np.median(widest)), float(best)\n\n\ndef fit_controller_v2a(feat200, labels200):\n    """v2a: logistic regression on the 11 retrieval features only (no question\n    embedding, which memorised families in v1). Threshold chosen on\n    leave-one-FAMILY-out out-of-fold probabilities."""\n    df = feat200.merge(labels200[["question_id", "template_family", "ft_direct_correct", "ft_rag_correct"]].rename(columns={"template_family": "_fam"}), on="question_id")\n    X = df[RETRIEVAL_FEATURES].to_numpy(float)\n    y = (df.ft_rag_correct & ~df.ft_direct_correct).astype(int).to_numpy()\n    fam = df["_fam"].to_numpy()\n    make = lambda: Pipeline([("s", StandardScaler()),\n                             ("lr", LogisticRegression(C=1.0, class_weight="balanced",\n                                                       max_iter=5000, random_state=42))])\n    oof = np.zeros(len(df))\n    for f in np.unique(fam):\n        tr, te = fam != f, fam == f\n        if y[tr].sum() == 0:\n            oof[te] = 0.0\n            continue\n        m = make().fit(X[tr], y[tr]); oof[te] = m.predict_proba(X[te])[:, 1]\n    tau, oof_acc = select_threshold(oof, df.ft_direct_correct.to_numpy(), df.ft_rag_correct.to_numpy(),\n                                    np.round(np.arange(0.05, 0.951, 0.01), 2))\n    model = make().fit(X, y)\n    return model, tau, oof_acc\n\n\ndef fit_controller_v2b(term_sim200, labels200):\n    """v2b: training-free policy-term router. Route to RAG when the question\'s best\n    E5 similarity to a policy metric name is >= tau. tau chosen on the 200 old\n    questions (single parameter, no family-specific learning)."""\n    df = term_sim200[["question_id", "term_similarity"]].merge(labels200, on="question_id")\n    tau, acc = select_threshold(df.term_similarity.to_numpy(), df.ft_direct_correct.to_numpy(),\n                                df.ft_rag_correct.to_numpy(), np.round(np.arange(0.70, 0.951, 0.005), 3))\n    return tau, acc\n'
LABELS_CSV = 'question_id,split,template_family,ft_direct_correct,ft_rag_correct\nB4A-FT-005-ENGLISH,validation,orders_average_tax_by_status,True,True\nB4A-FT-005-ARABIC,validation,orders_average_tax_by_status,True,True\nB4A-FT-005-FRENCH,validation,orders_average_tax_by_status,True,False\nB4A-FT-005-LEBANESE_ARABIZI,validation,orders_average_tax_by_status,True,True\nB4A-FT-010-ENGLISH,validation,orders_average_tax_by_status,True,True\nB4A-FT-010-ARABIC,validation,orders_average_tax_by_status,True,True\nB4A-FT-010-FRENCH,validation,orders_average_tax_by_status,True,True\nB4A-FT-010-LEBANESE_ARABIZI,validation,orders_average_tax_by_status,True,False\nB4A-FT-015-ENGLISH,validation,orders_average_tax_by_status,True,True\nB4A-FT-015-ARABIC,validation,orders_average_tax_by_status,True,True\nB4A-FT-015-FRENCH,validation,orders_average_tax_by_status,True,True\nB4A-FT-015-LEBANESE_ARABIZI,validation,orders_average_tax_by_status,True,False\nB4A-FT-020-ENGLISH,validation,orders_average_tax_by_status,True,True\nB4A-FT-020-ARABIC,validation,orders_average_tax_by_status,True,True\nB4A-FT-020-FRENCH,validation,orders_average_tax_by_status,True,True\nB4A-FT-020-LEBANESE_ARABIZI,validation,orders_average_tax_by_status,True,True\nB4A-FT-025-ENGLISH,validation,orders_average_tax_by_status,True,True\nB4A-FT-025-ARABIC,validation,orders_average_tax_by_status,True,True\nB4A-FT-025-FRENCH,validation,orders_average_tax_by_status,True,True\nB4A-FT-025-LEBANESE_ARABIZI,validation,orders_average_tax_by_status,True,False\nB4A-FT-030-ENGLISH,validation,orders_average_tax_by_status,True,True\nB4A-FT-030-ARABIC,validation,orders_average_tax_by_status,True,True\nB4A-FT-030-FRENCH,validation,orders_average_tax_by_status,True,True\nB4A-FT-030-LEBANESE_ARABIZI,validation,orders_average_tax_by_status,True,True\nB4A-FT-051-ENGLISH,validation,payment_method_count,True,True\nB4A-FT-051-ARABIC,validation,payment_method_count,True,True\nB4A-FT-051-FRENCH,validation,payment_method_count,True,False\nB4A-FT-051-LEBANESE_ARABIZI,validation,payment_method_count,True,True\nB4A-FT-055-ENGLISH,validation,payment_method_paid_total,True,True\nB4A-FT-055-ARABIC,validation,payment_method_paid_total,True,True\nB4A-FT-055-FRENCH,validation,payment_method_paid_total,True,True\nB4A-FT-055-LEBANESE_ARABIZI,validation,payment_method_paid_total,True,True\nB4A-FT-056-ENGLISH,validation,payment_method_count,True,True\nB4A-FT-056-ARABIC,validation,payment_method_count,True,True\nB4A-FT-056-FRENCH,validation,payment_method_count,True,True\nB4A-FT-056-LEBANESE_ARABIZI,validation,payment_method_count,True,True\nB4A-FT-060-ENGLISH,validation,payment_method_paid_total,True,True\nB4A-FT-060-ARABIC,validation,payment_method_paid_total,True,True\nB4A-FT-060-FRENCH,validation,payment_method_paid_total,True,True\nB4A-FT-060-LEBANESE_ARABIZI,validation,payment_method_paid_total,True,True\nB4A-FT-061-ENGLISH,validation,payment_method_count,True,True\nB4A-FT-061-ARABIC,validation,payment_method_count,True,True\nB4A-FT-061-FRENCH,validation,payment_method_count,True,True\nB4A-FT-061-LEBANESE_ARABIZI,validation,payment_method_count,True,True\nB4A-FT-065-ENGLISH,validation,payment_method_paid_total,True,True\nB4A-FT-065-ARABIC,validation,payment_method_paid_total,True,True\nB4A-FT-065-FRENCH,validation,payment_method_paid_total,True,True\nB4A-FT-065-LEBANESE_ARABIZI,validation,payment_method_paid_total,True,True\nB4A-FT-066-ENGLISH,validation,payment_method_count,True,True\nB4A-FT-066-ARABIC,validation,payment_method_count,True,True\nB4A-FT-066-FRENCH,validation,payment_method_count,True,True\nB4A-FT-066-LEBANESE_ARABIZI,validation,payment_method_count,True,True\nB4A-FT-070-ENGLISH,validation,payment_method_paid_total,True,True\nB4A-FT-070-ARABIC,validation,payment_method_paid_total,True,True\nB4A-FT-070-FRENCH,validation,payment_method_paid_total,True,True\nB4A-FT-070-LEBANESE_ARABIZI,validation,payment_method_paid_total,True,True\nB4A-FT-071-ENGLISH,validation,payment_method_count,True,True\nB4A-FT-071-ARABIC,validation,payment_method_count,True,True\nB4A-FT-071-FRENCH,validation,payment_method_count,True,False\nB4A-FT-071-LEBANESE_ARABIZI,validation,payment_method_count,True,True\nB4A-FT-075-ENGLISH,validation,payment_method_paid_total,True,True\nB4A-FT-075-ARABIC,validation,payment_method_paid_total,True,True\nB4A-FT-075-FRENCH,validation,payment_method_paid_total,True,True\nB4A-FT-075-LEBANESE_ARABIZI,validation,payment_method_paid_total,True,True\nB4A-FT-077-ENGLISH,validation,products_average_price_by_status,True,True\nB4A-FT-077-ARABIC,validation,products_average_price_by_status,True,True\nB4A-FT-077-FRENCH,validation,products_average_price_by_status,True,False\nB4A-FT-077-LEBANESE_ARABIZI,validation,products_average_price_by_status,True,True\nB4A-FT-079-ENGLISH,validation,products_inventory_value_by_status,False,True\nB4A-FT-079-ARABIC,validation,products_inventory_value_by_status,False,True\nB4A-FT-079-FRENCH,validation,products_inventory_value_by_status,False,True\nB4A-FT-079-LEBANESE_ARABIZI,validation,products_inventory_value_by_status,False,True\nB4A-FT-080-ENGLISH,validation,products_categories_by_status,True,True\nB4A-FT-080-ARABIC,validation,products_categories_by_status,True,True\nB4A-FT-080-FRENCH,validation,products_categories_by_status,True,True\nB4A-FT-080-LEBANESE_ARABIZI,validation,products_categories_by_status,True,False\nB4A-FT-082-ENGLISH,validation,products_average_price_by_status,True,True\nB4A-FT-082-ARABIC,validation,products_average_price_by_status,True,True\nB4A-FT-082-FRENCH,validation,products_average_price_by_status,True,True\nB4A-FT-082-LEBANESE_ARABIZI,validation,products_average_price_by_status,True,True\nB4A-FT-084-ENGLISH,validation,products_inventory_value_by_status,False,True\nB4A-FT-084-ARABIC,validation,products_inventory_value_by_status,False,False\nB4A-FT-084-FRENCH,validation,products_inventory_value_by_status,False,True\nB4A-FT-084-LEBANESE_ARABIZI,validation,products_inventory_value_by_status,False,False\nB4A-FT-085-ENGLISH,validation,products_categories_by_status,True,True\nB4A-FT-085-ARABIC,validation,products_categories_by_status,True,True\nB4A-FT-085-FRENCH,validation,products_categories_by_status,True,True\nB4A-FT-085-LEBANESE_ARABIZI,validation,products_categories_by_status,True,False\nB4A-FT-087-ENGLISH,validation,products_average_price_by_status,True,True\nB4A-FT-087-ARABIC,validation,products_average_price_by_status,True,True\nB4A-FT-087-FRENCH,validation,products_average_price_by_status,True,False\nB4A-FT-087-LEBANESE_ARABIZI,validation,products_average_price_by_status,True,True\nB4A-FT-089-ENGLISH,validation,products_inventory_value_by_status,True,True\nB4A-FT-089-ARABIC,validation,products_inventory_value_by_status,True,True\nB4A-FT-089-FRENCH,validation,products_inventory_value_by_status,True,True\nB4A-FT-089-LEBANESE_ARABIZI,validation,products_inventory_value_by_status,True,True\nB4A-FT-090-ENGLISH,validation,products_categories_by_status,True,True\nB4A-FT-090-ARABIC,validation,products_categories_by_status,True,True\nB4A-FT-090-FRENCH,validation,products_categories_by_status,True,True\nB4A-FT-090-LEBANESE_ARABIZI,validation,products_categories_by_status,True,False\nB4A-FT-003-ENGLISH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-003-ARABIC,final_test,orders_average_amount_by_status,True,True\nB4A-FT-003-FRENCH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-003-LEBANESE_ARABIZI,final_test,orders_average_amount_by_status,True,True\nB4A-FT-008-ENGLISH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-008-ARABIC,final_test,orders_average_amount_by_status,True,True\nB4A-FT-008-FRENCH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-008-LEBANESE_ARABIZI,final_test,orders_average_amount_by_status,True,True\nB4A-FT-013-ENGLISH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-013-ARABIC,final_test,orders_average_amount_by_status,True,True\nB4A-FT-013-FRENCH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-013-LEBANESE_ARABIZI,final_test,orders_average_amount_by_status,True,True\nB4A-FT-018-ENGLISH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-018-ARABIC,final_test,orders_average_amount_by_status,True,True\nB4A-FT-018-FRENCH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-018-LEBANESE_ARABIZI,final_test,orders_average_amount_by_status,True,True\nB4A-FT-023-ENGLISH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-023-ARABIC,final_test,orders_average_amount_by_status,True,True\nB4A-FT-023-FRENCH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-023-LEBANESE_ARABIZI,final_test,orders_average_amount_by_status,True,True\nB4A-FT-028-ENGLISH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-028-ARABIC,final_test,orders_average_amount_by_status,True,True\nB4A-FT-028-FRENCH,final_test,orders_average_amount_by_status,True,True\nB4A-FT-028-LEBANESE_ARABIZI,final_test,orders_average_amount_by_status,True,True\nB4A-FT-034-ENGLISH,final_test,payments_orders_by_status,True,True\nB4A-FT-034-ARABIC,final_test,payments_orders_by_status,True,True\nB4A-FT-034-FRENCH,final_test,payments_orders_by_status,True,True\nB4A-FT-034-LEBANESE_ARABIZI,final_test,payments_orders_by_status,True,True\nB4A-FT-039-ENGLISH,final_test,payments_orders_by_status,True,True\nB4A-FT-039-ARABIC,final_test,payments_orders_by_status,True,True\nB4A-FT-039-FRENCH,final_test,payments_orders_by_status,True,True\nB4A-FT-039-LEBANESE_ARABIZI,final_test,payments_orders_by_status,True,True\nB4A-FT-044-ENGLISH,final_test,payments_orders_by_status,True,True\nB4A-FT-044-ARABIC,final_test,payments_orders_by_status,True,True\nB4A-FT-044-FRENCH,final_test,payments_orders_by_status,True,True\nB4A-FT-044-LEBANESE_ARABIZI,final_test,payments_orders_by_status,True,True\nB4A-FT-049-ENGLISH,final_test,payments_orders_by_status,True,True\nB4A-FT-049-ARABIC,final_test,payments_orders_by_status,True,True\nB4A-FT-049-FRENCH,final_test,payments_orders_by_status,True,True\nB4A-FT-049-LEBANESE_ARABIZI,final_test,payments_orders_by_status,True,True\nB4A-FT-076-ENGLISH,final_test,products_count_by_status,True,True\nB4A-FT-076-ARABIC,final_test,products_count_by_status,True,True\nB4A-FT-076-FRENCH,final_test,products_count_by_status,True,True\nB4A-FT-076-LEBANESE_ARABIZI,final_test,products_count_by_status,True,True\nB4A-FT-081-ENGLISH,final_test,products_count_by_status,True,True\nB4A-FT-081-ARABIC,final_test,products_count_by_status,True,True\nB4A-FT-081-FRENCH,final_test,products_count_by_status,True,True\nB4A-FT-081-LEBANESE_ARABIZI,final_test,products_count_by_status,True,True\nB4A-FT-086-ENGLISH,final_test,products_count_by_status,True,True\nB4A-FT-086-ARABIC,final_test,products_count_by_status,True,True\nB4A-FT-086-FRENCH,final_test,products_count_by_status,True,True\nB4A-FT-086-LEBANESE_ARABIZI,final_test,products_count_by_status,True,True\nB4A-FT-091-ENGLISH,final_test,events_count_by_type,True,True\nB4A-FT-091-ARABIC,final_test,events_count_by_type,True,True\nB4A-FT-091-FRENCH,final_test,events_count_by_type,True,True\nB4A-FT-091-LEBANESE_ARABIZI,final_test,events_count_by_type,True,True\nB4A-FT-095-ENGLISH,final_test,events_top_products_by_type,False,True\nB4A-FT-095-ARABIC,final_test,events_top_products_by_type,True,True\nB4A-FT-095-FRENCH,final_test,events_top_products_by_type,True,True\nB4A-FT-095-LEBANESE_ARABIZI,final_test,events_top_products_by_type,True,True\nB4A-FT-096-ENGLISH,final_test,events_count_by_type,True,True\nB4A-FT-096-ARABIC,final_test,events_count_by_type,True,True\nB4A-FT-096-FRENCH,final_test,events_count_by_type,True,True\nB4A-FT-096-LEBANESE_ARABIZI,final_test,events_count_by_type,True,True\nB4A-FT-100-ENGLISH,final_test,events_top_products_by_type,False,True\nB4A-FT-100-ARABIC,final_test,events_top_products_by_type,True,True\nB4A-FT-100-FRENCH,final_test,events_top_products_by_type,True,True\nB4A-FT-100-LEBANESE_ARABIZI,final_test,events_top_products_by_type,True,True\nB4A-FT-101-ENGLISH,final_test,events_count_by_type,True,True\nB4A-FT-101-ARABIC,final_test,events_count_by_type,True,True\nB4A-FT-101-FRENCH,final_test,events_count_by_type,True,True\nB4A-FT-101-LEBANESE_ARABIZI,final_test,events_count_by_type,True,True\nB4A-FT-105-ENGLISH,final_test,events_top_products_by_type,False,True\nB4A-FT-105-ARABIC,final_test,events_top_products_by_type,True,False\nB4A-FT-105-FRENCH,final_test,events_top_products_by_type,True,True\nB4A-FT-105-LEBANESE_ARABIZI,final_test,events_top_products_by_type,True,True\nB4A-FT-106-ENGLISH,final_test,events_count_by_type,True,True\nB4A-FT-106-ARABIC,final_test,events_count_by_type,True,True\nB4A-FT-106-FRENCH,final_test,events_count_by_type,True,True\nB4A-FT-106-LEBANESE_ARABIZI,final_test,events_count_by_type,True,True\nB4A-FT-110-ENGLISH,final_test,events_top_products_by_type,False,True\nB4A-FT-110-ARABIC,final_test,events_top_products_by_type,True,False\nB4A-FT-110-FRENCH,final_test,events_top_products_by_type,True,True\nB4A-FT-110-LEBANESE_ARABIZI,final_test,events_top_products_by_type,True,True\nB4A-FT-111-ENGLISH,final_test,events_count_by_type,True,True\nB4A-FT-111-ARABIC,final_test,events_count_by_type,True,True\nB4A-FT-111-FRENCH,final_test,events_count_by_type,True,True\nB4A-FT-111-LEBANESE_ARABIZI,final_test,events_count_by_type,True,True\nB4A-FT-115-ENGLISH,final_test,events_top_products_by_type,False,True\nB4A-FT-115-ARABIC,final_test,events_top_products_by_type,True,True\nB4A-FT-115-FRENCH,final_test,events_top_products_by_type,True,True\nB4A-FT-115-LEBANESE_ARABIZI,final_test,events_top_products_by_type,True,True\nB4A-FT-116-ENGLISH,final_test,events_count_by_type,True,True\nB4A-FT-116-ARABIC,final_test,events_count_by_type,True,True\nB4A-FT-116-FRENCH,final_test,events_count_by_type,True,True\nB4A-FT-116-LEBANESE_ARABIZI,final_test,events_count_by_type,True,True\nB4A-FT-120-ENGLISH,final_test,events_top_products_by_type,False,True\nB4A-FT-120-ARABIC,final_test,events_top_products_by_type,True,False\nB4A-FT-120-FRENCH,final_test,events_top_products_by_type,True,True\nB4A-FT-120-LEBANESE_ARABIZI,final_test,events_top_products_by_type,True,True\n'
exec(CTRL_SRC)


# ------------------------------------------------------------
# D1 — FREEZE THE CONTROLLERS (runs BEFORE any PD-Test generation)
# ------------------------------------------------------------
import io, json, hashlib, datetime, joblib
LABELS_200 = pd.read_csv(io.StringIO(LABELS_CSV))
assert len(LABELS_200) == 200

feat_old = pd.read_csv(REV / "B5_controller_features.csv")
feat_old200 = feat_old[feat_old.question_id.isin(LABELS_200.question_id)].copy()
assert len(feat_old200) == 200

def retrieval_features(questions):
    rows = []
    for q in questions:
        r = retrieve_chunks(q, top_k=6)
        s = ([float(c["retrieval_score"]) for c in r] + [0.0] * 6)[:6]
        rows.append(s + [s[0], float(np.mean(s)), float(np.std(s)), s[0] - s[1], float(len(q))])
    return pd.DataFrame(rows, columns=RETRIEVAL_FEATURES)

def encode(texts):
    return embedding_model.encode(["query: " + t for t in texts], batch_size=32, convert_to_numpy=True,
                                  normalize_embeddings=True, show_progress_bar=False)

term_emb = encode(POLICY_METRIC_NAMES)
old_q = feat_old200.merge(custom_dataset_df[["question_id", "question"]], on="question_id")
term_old = pd.DataFrame({"question_id": old_q.question_id,
                         "term_similarity": (encode(old_q.question.astype(str).tolist()) @ term_emb.T).max(axis=1)})

# Fit v2a and v2b on OLD data only
v2a_model, v2a_tau, v2a_oof_acc = fit_controller_v2a(feat_old200, LABELS_200)
v2b_tau, v2b_acc = fit_controller_v2b(term_old, LABELS_200)

# PD-Test features and routing decisions (no PD outcome exists yet)
pd_q = pd_df["question"].astype(str).tolist()
pd_ret = retrieval_features(pd_q)
pd_emb = encode(pd_q)
pd_term = (pd_emb @ term_emb.T).max(axis=1)
controller_v1 = joblib.load(sorted(ROOT.rglob("strict_adaptive_controller.joblib"))[0])
p_v1 = controller_v1.predict_proba(np.concatenate([pd_emb, pd_ret.to_numpy(np.float32)], axis=1))[:, 1]
p_v2a = v2a_model.predict_proba(pd_ret[RETRIEVAL_FEATURES].to_numpy(float))[:, 1]

routes = pd_df[["question_id", "intent_id", "template_family", "question_type", "language"]].copy()
routes = pd.concat([routes.reset_index(drop=True), pd_ret.reset_index(drop=True)], axis=1)
routes["term_similarity"] = pd_term
routes["p_v1"], routes["route_v1"] = p_v1, p_v1 >= 0.93
routes["p_v2a"], routes["route_v2a"] = p_v2a, p_v2a >= v2a_tau
routes["route_v2b"] = pd_term >= v2b_tau
routes_path = REV / "D1_frozen_controller_routes.csv"
routes.to_csv(routes_path, index=False)
joblib.dump(v2a_model, REV / "D1_controller_v2a.joblib")

manifest = {
    "frozen_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "pd_test_generations_before_freeze": int(sum(1 for p in REV.glob("D2_*.jsonl")) + sum(1 for p in REV.glob("D3_*.jsonl"))),
    "v1_threshold": 0.93, "v2a_threshold": v2a_tau, "v2a_oof_accuracy_old200": v2a_oof_acc,
    "v2b_threshold": v2b_tau, "v2b_accuracy_old200": v2b_acc,
    "routes_sha256": hashlib.sha256(routes_path.read_bytes()).hexdigest(),
    "controller_source_sha256": hashlib.sha256(CTRL_SRC.encode()).hexdigest(),
}
(REV / "D1_freeze_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print(routes.groupby("question_type")[["route_v1", "route_v2a", "route_v2b"]].mean())
assert manifest["pd_test_generations_before_freeze"] == 0, "PD generations already exist — tell Claude."
print("CONTROLLERS FROZEN. Now run D2 and D3.")

{
  "frozen_utc": "2026-10-06T20:38:26.350312+00:00",
  "pd_test_generations_before_freeze": 0,
  "v1_threshold": 0.93,
  "v2a_threshold": 0.15,
  "v2a_oof_accuracy_old200": 0.975,
  "v2b_threshold": 0.83,
  "v2b_accuracy_old200": 0.97,
  "routes_sha256": "09d92cdf7c91df378f465cf0e72bdf366c88816e0e519a75fbec186b934ed8e0",
  "controller_source_sha256": "f2a1aadf4fa07d5fb7406544c85090d2f03546d1bb2da2c7cd90cab04d3497d5"
}
                  route_v1  route_v2a  route_v2b
question_type                                   
explicit_control       0.0   0.359375    0.46875
policy_dependent       0.0   0.296875    0.84375
CONTROLLERS FROZEN. Now run D2 and D3.


In [21]:
# D2 — Fine-tuned Qwen, direct (no RAG)
def gen_direct(row):
    out = generate_sql(question=row["question"], use_rag=False)
    return {"sql": out["sql"], "latency_seconds": out["latency_seconds"]}
assert (REV / "D1_freeze_manifest.json").exists(), "Run D1 (freeze) first."
d2 = run_experiment("D2_pd_finetuned_direct", pd_df.assign(split="policy_test"), gen_direct, adapter=True)

D2_pd_finetuned_direct: 0 already done, 128 to run
[001/128] B4A-PD-P01-ENGLISH correct=False
[002/128] B4A-PD-P01-FRENCH correct=False
[003/128] B4A-PD-P01-ARABIC correct=False
[004/128] B4A-PD-P01-LEBANESE_ARABIZI correct=False
[005/128] B4A-PD-P02-ENGLISH correct=False
[006/128] B4A-PD-P02-FRENCH correct=False
[007/128] B4A-PD-P02-ARABIC correct=False
[008/128] B4A-PD-P02-LEBANESE_ARABIZI correct=False
[009/128] B4A-PD-P03-ENGLISH correct=False
[010/128] B4A-PD-P03-FRENCH correct=False
[011/128] B4A-PD-P03-ARABIC correct=False
[012/128] B4A-PD-P03-LEBANESE_ARABIZI correct=False
[013/128] B4A-PD-P04-ENGLISH correct=False
[014/128] B4A-PD-P04-FRENCH correct=False
[015/128] B4A-PD-P04-ARABIC correct=False
[016/128] B4A-PD-P04-LEBANESE_ARABIZI correct=False
[017/128] B4A-PD-P05-ENGLISH correct=False
[018/128] B4A-PD-P05-FRENCH correct=False
[019/128] B4A-PD-P05-ARABIC correct=False
[020/128] B4A-PD-P05-LEBANESE_ARABIZI correct=False
[021/128] B4A-PD-P06-ENGLISH correct=False
[022/128] B

In [22]:
# D3 — Fine-tuned Qwen + always-on RAG
def gen_rag(row):
    out = generate_sql(question=row["question"], use_rag=True)
    return {"sql": out["sql"], "latency_seconds": out["latency_seconds"]}
d3 = run_experiment("D3_pd_finetuned_rag", pd_df.assign(split="policy_test"), gen_rag, adapter=True)

D3_pd_finetuned_rag: 0 already done, 128 to run
[001/128] B4A-PD-P01-ENGLISH correct=True
[002/128] B4A-PD-P01-FRENCH correct=True
[003/128] B4A-PD-P01-ARABIC correct=True
[004/128] B4A-PD-P01-LEBANESE_ARABIZI correct=True
[005/128] B4A-PD-P02-ENGLISH correct=False
[006/128] B4A-PD-P02-FRENCH correct=False
[007/128] B4A-PD-P02-ARABIC correct=False
[008/128] B4A-PD-P02-LEBANESE_ARABIZI correct=False
[009/128] B4A-PD-P03-ENGLISH correct=True
[010/128] B4A-PD-P03-FRENCH correct=True
[011/128] B4A-PD-P03-ARABIC correct=True
[012/128] B4A-PD-P03-LEBANESE_ARABIZI correct=True
[013/128] B4A-PD-P04-ENGLISH correct=True
[014/128] B4A-PD-P04-FRENCH correct=True
[015/128] B4A-PD-P04-ARABIC correct=True
[016/128] B4A-PD-P04-LEBANESE_ARABIZI correct=True
[017/128] B4A-PD-P05-ENGLISH correct=True
[018/128] B4A-PD-P05-FRENCH correct=True
[019/128] B4A-PD-P05-ARABIC correct=True
[020/128] B4A-PD-P05-LEBANESE_ARABIZI correct=True
[021/128] B4A-PD-P06-ENGLISH correct=False
[022/128] B4A-PD-P06-FRENCH co

In [ ]:
# D4 (optional) — Original Qwen (no fine-tuning), zero-shot, on the PD-Test
d4 = run_experiment("D4_pd_original_zeroshot", pd_df.assign(split="policy_test"), gen_direct, adapter=False)

In [23]:
# D5 — Package PD results for Claude
import zipfile, datetime
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
zp = REV / f"pd_results_{stamp}.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(REV.glob("D*")):
        if p.suffix in {".csv", ".jsonl", ".npy"}:
            z.write(p, p.name)
print("Created:", zp)

Created: /content/drive/MyDrive/Build4All_Research/Revision_Experiments/pd_results_20261006_2121.zip


In [24]:
# FIX — restore the evaluator's datetime and re-run ONLY the affected questions
from datetime import date, datetime
assert normalize_evaluation_value(5) == "5"

def gen_direct(row):
    out = generate_sql(question=row["question"], use_rag=False)
    return {"sql": out["sql"], "latency_seconds": out["latency_seconds"]}
def gen_rag(row):
    out = generate_sql(question=row["question"], use_rag=True)
    return {"sql": out["sql"], "latency_seconds": out["latency_seconds"]}

for name in ["D2_pd_finetuned_direct", "D3_pd_finetuned_rag"]:
    p = REV / f"{name}.jsonl"
    lines = p.read_text(encoding="utf-8").splitlines()
    keep = [l for l in lines if "isinstance() arg 2" not in l]
    print(name, "removed bugged records:", len(lines) - len(keep))
    p.write_text("\n".join(keep) + "\n", encoding="utf-8")

assert (REV / "D1_freeze_manifest.json").exists()
d2 = run_experiment("D2_pd_finetuned_direct", pd_df.assign(split="policy_test"), gen_direct, adapter=True)
d3 = run_experiment("D3_pd_finetuned_rag", pd_df.assign(split="policy_test"), gen_rag, adapter=True)

D2_pd_finetuned_direct removed bugged records: 68
D3_pd_finetuned_rag removed bugged records: 68
D2_pd_finetuned_direct: 60 already done, 68 to run
[061/128] B4A-PD-P02-ENGLISH correct=False
[062/128] B4A-PD-P02-FRENCH correct=False
[063/128] B4A-PD-P02-ARABIC correct=False
[064/128] B4A-PD-P02-LEBANESE_ARABIZI correct=False
[065/128] B4A-PD-P06-ENGLISH correct=False
[066/128] B4A-PD-P06-FRENCH correct=False
[067/128] B4A-PD-P06-ARABIC correct=True
[068/128] B4A-PD-P06-LEBANESE_ARABIZI correct=False
[069/128] B4A-PD-P11-ENGLISH correct=False
[070/128] B4A-PD-P11-FRENCH correct=False
[071/128] B4A-PD-P11-ARABIC correct=False
[072/128] B4A-PD-P11-LEBANESE_ARABIZI correct=False
[073/128] B4A-PD-P12-ENGLISH correct=False
[074/128] B4A-PD-P12-FRENCH correct=False
[075/128] B4A-PD-P12-ARABIC correct=False
[076/128] B4A-PD-P12-LEBANESE_ARABIZI correct=False
[077/128] B4A-PD-P13-ENGLISH correct=False
[078/128] B4A-PD-P13-FRENCH correct=False
[079/128] B4A-PD-P13-ARABIC correct=False
[080/128] 